In [204]:
import time
import pandas as pd
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import spacy
import torch
import torch.nn as nn
import torch.optim as optim

### Dataset Creation

In [205]:
df = pd.read_csv("train_with_task_type.csv")
task_types = set(df["task_type"])

X, y = df["prompt"], df["task_type"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)

### Pipeline #1: TF-IDF + Naive Bayes

In [206]:
model = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])

start_train = time.time()
model.fit(X_train, y_train)
end_train = time.time()
print(f"Training Time: {end_train - start_train:.3f}s") # training time

start_inf = time.time()
y_pred = model.predict(X_test)
end_inf = time.time()
print(f"Inference Time: {end_inf - start_inf:.3f}s") # inference time

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

matrix = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{matrix}")

Training Time: 0.134s
Inference Time: 0.026s
Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1-Score:  1.000

Confusion Matrix:
[[322   0   0   0   0   0]
 [  0 293   0   0   0   0]
 [  0   0 347   0   0   0]
 [  0   0   0 308   0   0]
 [  0   0   0   0 306   0]
 [  0   0   0   0   0 324]]


### Pipeline #2: Word2Vec + Neural Network

In [207]:
# run this command to load pretrained word2vec embedding model: 
# !python3 -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md")

""" creating class indicies """
classes = list(set(y))
ctoi, itoc = {}, {}
for i in range(len(classes)):
    ctoi[classes[i]] = i
    itoc[i] = classes[i]

start_nlp = time.time()
""" creating document vectors via mean pooling """
nlpX_train = np.array([
    doc.vector
    for doc in nlp.pipe(X_train, batch_size=100, disable=["tagger", "parser", "ner"])
])
nlpX_test = np.array([
    doc.vector
    for doc in nlp.pipe(X_test, batch_size=100, disable=["tagger", "parser", "ner"])
])

nlpy_train = np.array([
    ctoi[text] for text in y_train
])

nlpy_test = np.array([
    ctoi[text] for text in y_test
])
end_nlp = time.time()

nlpX_train, nlpX_test = torch.tensor(nlpX_train), torch.tensor(nlpX_test)
nlpy_train, nlpy_test = torch.tensor(nlpy_train, dtype=torch.long), torch.tensor(nlpy_test, dtype=torch.long)
    
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(300, 300),
            nn.ReLU(),
            nn.Linear(300, len(classes))
        )  
    
    def forward(self, x):
        return self.model(x)

model = NeuralNetwork()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

""" training loop """
epochs = 500
for i in range(epochs):
    optimizer.zero_grad()

    out = model(nlpX_train)
    loss = criterion(out, nlpy_train)

    loss.backward()
    optimizer.step()
    # print(f"Loss: {loss:.2f}")

""" evaluating accuracy """
with torch.no_grad():
    out = model(nlpX_test)
    prediction = torch.argmax(out, dim=1)

accuracy = (prediction == nlpy_test).float().mean()
print("Accuracy:", accuracy.item())

for pred, true in zip(prediction[:10], nlpy_test[:10]):
    print(f"Pred: {[itoc[pred.item()]]} | True: {[itoc[true.item()]]}")

print(f"Data Preprocessing Time: {end_nlp - start_nlp}s")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Accuracy: 0.9984210729598999
Pred: ['bit_manipulation'] | True: ['bit_manipulation']
Pred: ['unit_conversion'] | True: ['unit_conversion']
Pred: ['symbol_transform'] | True: ['symbol_transform']
Pred: ['symbol_transform'] | True: ['symbol_transform']
Pred: ['cipher_text'] | True: ['cipher_text']
Pred: ['gravity'] | True: ['gravity']
Pred: ['roman'] | True: ['roman']
Pred: ['cipher_text'] | True: ['cipher_text']
Pred: ['symbol_transform'] | True: ['symbol_transform']
Pred: ['gravity'] | True: ['gravity']
Data Preprocessing Time: 43.56122612953186s
